# LCC-FASD — Local CPU/XPU Performance Benchmark (Complete)

Notebook này đo hiệu năng local từ đầu đến cuối, không train full model.

Mục tiêu:
- kiểm tra Python / PyTorch / CPU / RAM / Intel XPU;
- kiểm tra XPU bằng tensor operation có synchronize;
- tự tìm training split của LCC-FASD;
- benchmark CNN cố định bằng forward + backward + optimizer step;
- timing XPU đúng với `torch.xpu.synchronize()`;
- đo **compute-only** và **end-to-end**;
- benchmark batch size, DataLoader workers, CPU threads, resolution, augmentation;
- re-test một tập cấu hình cuối;
- tự chọn cấu hình theo **end-to-end throughput**;
- xuất CSV / JSON / TXT.

**Không có cell nào cài lại torch/torchvision.** Notebook dành cho kernel `.venv-xpu` đã có PyTorch XPU.

## 1. Setup — không thay đổi PyTorch build

In [1]:
import os, gc, time, json, platform, statistics, warnings
from pathlib import Path

import torch
import torchvision
import torch.nn as nn
import pandas as pd
import psutil
from PIL import Image
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("OS:", platform.platform())
print("CPU logical processors:", os.cpu_count())
print("RAM total (GB):", round(psutil.virtual_memory().total / (1024**3), 2))
print("PyTorch CPU threads:", torch.get_num_threads())
print("CUDA available:", torch.cuda.is_available())
print("XPU available:", torch.xpu.is_available())
print("XPU device count:", torch.xpu.device_count())

if not torch.xpu.is_available():
    raise RuntimeError("XPU không khả dụng trong kernel hiện tại. Dừng benchmark để tránh chạy nhầm CPU.")

print("XPU device:", torch.xpu.get_device_name(0))
DEVICE = torch.device("xpu")
print("Benchmark device:", DEVICE)

Python: 3.11.9
PyTorch: 2.14.0+xpu
Torchvision: 0.29.0+xpu
OS: Linux-7.0.0-22-generic-x86_64-with-glibc2.43
CPU logical processors: 14
RAM total (GB): 14.55
PyTorch CPU threads: 12
CUDA available: False
XPU available: True
XPU device count: 1
XPU device: Intel(R) Graphics
Benchmark device: xpu


## 2. XPU smoke test — bắt buộc pass trước benchmark

In [5]:
torch.manual_seed(123)
x = torch.randn(1024, 1024, device=DEVICE)
y = torch.randn(1024, 1024, device=DEVICE)

torch.xpu.synchronize()
t0 = time.perf_counter()
z = x @ y
torch.xpu.synchronize()
elapsed = time.perf_counter() - t0

print("Device:", z.device)
print("Shape:", tuple(z.shape))
print("Elapsed (s):", round(elapsed, 6))
print("Finite:", bool(torch.isfinite(z).all().item()))

del x, y, z
gc.collect()

Device: xpu:0
Shape: (1024, 1024)
Elapsed (s): 0.000344
Finite: True


0

## 3. Dataset discovery — chỉ dùng training split

In [7]:
DATASET_ROOT = Path("../dataset")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {DATASET_ROOT.resolve()}. "
        "Hãy mở notebook từ project root chứa dataset/."
    )

def find_class_roots(root):
    found = []
    for p in root.rglob("*"):
        if p.is_dir():
            try:
                subdirs = {x.name.lower() for x in p.iterdir() if x.is_dir()}
            except PermissionError:
                continue
            if {"real", "spoof"}.issubset(subdirs):
                found.append(p)
    return found

class_roots = find_class_roots(DATASET_ROOT)
if not class_roots:
    raise RuntimeError("Không tìm thấy thư mục có cả real/ và spoof/.")

def train_score(p):
    n = p.name.lower()
    return (100 if "training" in n else 0, 50 if "train" in n else 0, len(str(p)))

TRAIN_ROOT = sorted(class_roots, key=train_score, reverse=True)[0]
preview = ImageFolder(str(TRAIN_ROOT))

print("Dataset root:", DATASET_ROOT.resolve())
print("Candidate class roots:")
for p in class_roots:
    print(" -", p)
print("TRAIN_ROOT:", TRAIN_ROOT.resolve())
print("Classes:", preview.classes)
print("Mapping:", preview.class_to_idx)
print("Training images:", len(preview))

if preview.class_to_idx != {"real": 0, "spoof": 1}:
    print("WARNING: mapping khác {'real': 0, 'spoof': 1}; benchmark vẫn chạy nhưng hãy kiểm tra dataset.")

Dataset root: /home/dainguyen1506/Documents/face-anti-spoofing/dataset
Candidate class roots:
 - ../dataset/LCC_FASD/LCC_FASD_evaluation
 - ../dataset/LCC_FASD/LCC_FASD_development
 - ../dataset/LCC_FASD/LCC_FASD_training
TRAIN_ROOT: /home/dainguyen1506/Documents/face-anti-spoofing/dataset/LCC_FASD/LCC_FASD_training
Classes: ['real', 'spoof']
Mapping: {'real': 0, 'spoof': 1}
Training images: 8299


## 4. Fixed benchmark CNN

Model benchmark cố định, không phải model FAS cuối cùng. Mục đích là giữ workload giống nhau giữa các phép đo.

In [8]:
class BenchmarkCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(self.features(x))
        return self.classifier(torch.flatten(x, 1))

print(BenchmarkCNN())

BenchmarkCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNor

## 5. DataLoader + timing functions

Có hai phép đo:
- **compute-only:** batch và CPU→device transfer nằm ngoài timer; đo forward + backward + optimizer.
- **end-to-end:** timer bao gồm lấy batch, preprocessing, CPU→device transfer, forward, backward và optimizer.

Với XPU, synchronize trước và sau vùng đo để tránh số liệu giả do execution bất đồng bộ.

In [9]:
def make_transform(img_size, use_augmentation=True):
    if use_augmentation:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(
                brightness=0.10, contrast=0.10,
                saturation=0.10, hue=0.02
            ),
            transforms.ToTensor(),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ])

def make_loader(batch_size, num_workers, img_size, use_augmentation=True):
    ds = ImageFolder(
        str(TRAIN_ROOT),
        transform=make_transform(img_size, use_augmentation)
    )
    kwargs = dict(
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=False,
        drop_last=True,
    )
    if num_workers > 0:
        kwargs["persistent_workers"] = True
    return ds, DataLoader(ds, **kwargs)

def sync_device(device):
    if device.type == "xpu":
        torch.xpu.synchronize()

def train_step(model, optimizer, criterion, x, y):
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
    return loss

def benchmark_compute(
    batch_size, num_workers, img_size, cpu_threads,
    warmup_batches=10, measure_batches=100,
    use_augmentation=True, device=DEVICE
):
    gc.collect()
    torch.set_num_threads(cpu_threads)
    ds, loader = make_loader(batch_size, num_workers, img_size, use_augmentation)

    model = BenchmarkCNN().to(device).train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss()
    it = iter(loader)

    for _ in range(warmup_batches):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader)
            x, y = next(it)
        x, y = x.to(device), y.to(device)
        train_step(model, optimizer, criterion, x, y)

    sync_device(device)
    times = []

    for _ in range(measure_batches):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader)
            x, y = next(it)
        x, y = x.to(device), y.to(device)

        sync_device(device)
        t0 = time.perf_counter()
        train_step(model, optimizer, criterion, x, y)
        sync_device(device)
        times.append(time.perf_counter() - t0)

    sec = statistics.mean(times)
    ips = batch_size / sec

    result = {
        "device": str(device), "mode": "compute_only",
        "batch_size": batch_size, "num_workers": num_workers,
        "img_size": img_size, "cpu_threads": cpu_threads,
        "augmentation": "ON" if use_augmentation else "OFF",
        "dataset_images": len(ds), "seconds_per_batch": sec,
        "images_per_sec": ips, "estimated_epoch_min": len(ds) / ips / 60,
    }

    del model, optimizer, criterion, loader, ds
    gc.collect()
    if device.type == "xpu":
        torch.xpu.empty_cache()
    return result

def benchmark_end_to_end(
    batch_size, num_workers, img_size, cpu_threads,
    warmup_batches=10, measure_batches=60,
    use_augmentation=True, device=DEVICE
):
    gc.collect()
    torch.set_num_threads(cpu_threads)
    ds, loader = make_loader(batch_size, num_workers, img_size, use_augmentation)

    model = BenchmarkCNN().to(device).train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss()
    it = iter(loader)

    for _ in range(warmup_batches):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader)
            x, y = next(it)
        x, y = x.to(device), y.to(device)
        train_step(model, optimizer, criterion, x, y)

    sync_device(device)
    times = []

    for _ in range(measure_batches):
        sync_device(device)
        t0 = time.perf_counter()

        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader)
            x, y = next(it)

        x, y = x.to(device), y.to(device)
        train_step(model, optimizer, criterion, x, y)

        sync_device(device)
        times.append(time.perf_counter() - t0)

    sec = statistics.mean(times)
    ips = batch_size / sec

    result = {
        "device": str(device), "mode": "end_to_end",
        "batch_size": batch_size, "num_workers": num_workers,
        "img_size": img_size, "cpu_threads": cpu_threads,
        "augmentation": "ON" if use_augmentation else "OFF",
        "dataset_images": len(ds), "seconds_per_batch": sec,
        "images_per_sec": ips, "estimated_epoch_min": len(ds) / ips / 60,
    }

    del model, optimizer, criterion, loader, ds
    gc.collect()
    if device.type == "xpu":
        torch.xpu.empty_cache()
    return result

## 6. Quick sanity benchmark

In [10]:
SANITY = benchmark_end_to_end(
    batch_size=32, num_workers=2, img_size=128, cpu_threads=8,
    warmup_batches=5, measure_batches=20, use_augmentation=True
)
print(
    f"Sanity | {SANITY['device']} | "
    f"{SANITY['images_per_sec']:.2f} images/s | "
    f"{SANITY['estimated_epoch_min']:.2f} min/epoch"
)

Sanity | xpu | 199.46 images/s | 0.69 min/epoch


## 7. Benchmark A — batch size × DataLoader workers

Giữ image 128, CPU threads 8, augmentation ON. Đây là end-to-end.

In [ ]:
results_A = []
for batch in [16, 32, 64, 128]:
    for workers in [i for i in range(0, os.cpu_count() - 1) if i % 2 == 0]:
        print(f"Running A | batch={batch}, workers={workers}, threads=8, img=128")
        try:
            r = benchmark_end_to_end(
                batch_size=batch, num_workers=workers, img_size=128,
                cpu_threads=8, warmup_batches=10, measure_batches=60,
                use_augmentation=True
            )
            r["benchmark"] = "A_batch_workers"
            results_A.append(r)
            print(f"  {r['images_per_sec']:.2f} images/s | {r['estimated_epoch_min']:.2f} min/epoch")
        except Exception as e:
            print("  FAILED:", repr(e))

df_A = pd.DataFrame(results_A)
display(df_A.sort_values("images_per_sec", ascending=False) if len(df_A) else df_A)

Running A | batch=16, workers=13, threads=8, img=128
  200.13 images/s | 0.69 min/epoch
Running A | batch=16, workers=7, threads=8, img=128
  207.92 images/s | 0.67 min/epoch
Running A | batch=16, workers=0, threads=8, img=128
  60.76 images/s | 2.28 min/epoch
Running A | batch=32, workers=13, threads=8, img=128
  203.72 images/s | 0.68 min/epoch
Running A | batch=32, workers=7, threads=8, img=128


KeyboardInterrupt: 

## 8. Benchmark B — CPU threads

Lấy cấu hình nhanh nhất từ A, sau đó test 4/8/12/14 threads nếu máy có đủ logical processors.

In [12]:
if len(df_A) == 0:
    raise RuntimeError("Benchmark A không có kết quả.")

best_A = df_A.loc[df_A["images_per_sec"].idxmax()]
BEST_BATCH_A = int(best_A["batch_size"])
BEST_WORKERS_A = int(best_A["num_workers"])

thread_candidates = [t for t in [4, 8, 12, 14] if t <= (os.cpu_count() or 1)]
results_B = []

for threads in thread_candidates:
    print(
        f"Running B | batch={BEST_BATCH_A}, workers={BEST_WORKERS_A}, "
        f"threads={threads}, img=128"
    )
    try:
        r = benchmark_end_to_end(
            batch_size=BEST_BATCH_A, num_workers=BEST_WORKERS_A,
            img_size=128, cpu_threads=threads,
            warmup_batches=10, measure_batches=60,
            use_augmentation=True
        )
        r["benchmark"] = "B_cpu_threads"
        results_B.append(r)
        print(f"  {r['images_per_sec']:.2f} images/s | {r['estimated_epoch_min']:.2f} min/epoch")
    except Exception as e:
        print("  FAILED:", repr(e))

df_B = pd.DataFrame(results_B)
display(df_B.sort_values("images_per_sec", ascending=False) if len(df_B) else df_B)

Running B | batch=32, workers=4, threads=4, img=128
  188.90 images/s | 0.73 min/epoch
Running B | batch=32, workers=4, threads=8, img=128
  188.13 images/s | 0.74 min/epoch
Running B | batch=32, workers=4, threads=12, img=128
  208.06 images/s | 0.66 min/epoch
Running B | batch=32, workers=4, threads=14, img=128
  226.21 images/s | 0.61 min/epoch


,device,mode,batch_size,num_workers,img_size,cpu_threads,augmentation,dataset_images,seconds_per_batch,images_per_sec,estimated_epoch_min,benchmark
3,xpu,end_to_end,32,4,128,14,ON,8299,0.141460,226.212929,0.611445,B_cpu_threads
2,xpu,end_to_end,32,4,128,12,ON,8299,0.153804,208.057580,0.664800,B_cpu_threads
0,xpu,end_to_end,32,4,128,4,ON,8299,0.169406,188.895503,0.732239,B_cpu_threads
1,xpu,end_to_end,32,4,128,8,ON,8299,0.170097,188.127939,0.735227,B_cpu_threads


## 9. Benchmark C — input resolution

Dùng cấu hình tốt nhất hiện tại; test 96 / 128 / 160.

In [13]:
best_B = df_B.loc[df_B["images_per_sec"].idxmax()] if len(df_B) else best_A
BEST_BATCH = int(best_B["batch_size"])
BEST_WORKERS = int(best_B["num_workers"])
BEST_THREADS = int(best_B["cpu_threads"])

results_C = []
for size in [96, 128, 160, 256]:
    print(
        f"Running C | batch={BEST_BATCH}, workers={BEST_WORKERS}, "
        f"threads={BEST_THREADS}, img={size}"
    )
    try:
        r = benchmark_end_to_end(
            batch_size=BEST_BATCH, num_workers=BEST_WORKERS,
            img_size=size, cpu_threads=BEST_THREADS,
            warmup_batches=10, measure_batches=60,
            use_augmentation=True
        )
        r["benchmark"] = "C_resolution"
        results_C.append(r)
        print(f"  {r['images_per_sec']:.2f} images/s | {r['estimated_epoch_min']:.2f} min/epoch")
    except Exception as e:
        print("  FAILED:", repr(e))

df_C = pd.DataFrame(results_C)
display(df_C.sort_values("images_per_sec", ascending=False) if len(df_C) else df_C)

Running C | batch=32, workers=4, threads=14, img=96
  255.19 images/s | 0.54 min/epoch
Running C | batch=32, workers=4, threads=14, img=128
  215.38 images/s | 0.64 min/epoch
Running C | batch=32, workers=4, threads=14, img=160
  171.40 images/s | 0.81 min/epoch
Running C | batch=32, workers=4, threads=14, img=256
  58.18 images/s | 2.38 min/epoch


,device,mode,batch_size,num_workers,img_size,cpu_threads,augmentation,dataset_images,seconds_per_batch,images_per_sec,estimated_epoch_min,benchmark
0,xpu,end_to_end,32,4,96,14,ON,8299,0.125397,255.189487,0.542016,C_resolution
1,xpu,end_to_end,32,4,128,14,ON,8299,0.148572,215.384095,0.642186,C_resolution
2,xpu,end_to_end,32,4,160,14,ON,8299,0.186697,171.400995,0.806977,C_resolution
3,xpu,end_to_end,32,4,256,14,ON,8299,0.549986,58.183336,2.377256,C_resolution


## 10. Benchmark D — augmentation ON vs OFF

In [14]:
best_C = df_C.loc[df_C["images_per_sec"].idxmax()] if len(df_C) else best_B
CUR_BATCH = int(best_C["batch_size"])
CUR_WORKERS = int(best_C["num_workers"])
CUR_THREADS = int(best_C["cpu_threads"])
CUR_SIZE = int(best_C["img_size"])

results_D = []
for aug in [False, True]:
    label = "ON" if aug else "OFF"
    print(
        f"Running D | aug={label} | batch={CUR_BATCH}, workers={CUR_WORKERS}, "
        f"threads={CUR_THREADS}, img={CUR_SIZE}"
    )
    try:
        r = benchmark_end_to_end(
            batch_size=CUR_BATCH, num_workers=CUR_WORKERS,
            img_size=CUR_SIZE, cpu_threads=CUR_THREADS,
            warmup_batches=10, measure_batches=60,
            use_augmentation=aug
        )
        r["benchmark"] = "D_augmentation"
        results_D.append(r)
        print(f"  {r['images_per_sec']:.2f} images/s | {r['estimated_epoch_min']:.2f} min/epoch")
    except Exception as e:
        print("  FAILED:", repr(e))

df_D = pd.DataFrame(results_D)
display(df_D if len(df_D) else df_D)

Running D | aug=OFF | batch=32, workers=4, threads=14, img=96
  270.72 images/s | 0.51 min/epoch
Running D | aug=ON | batch=32, workers=4, threads=14, img=96
  274.37 images/s | 0.50 min/epoch


,device,mode,batch_size,num_workers,img_size,cpu_threads,augmentation,dataset_images,seconds_per_batch,images_per_sec,estimated_epoch_min,benchmark
0,xpu,end_to_end,32,4,96,14,OFF,8299,0.118201,270.724447,0.510913,D_augmentation
1,xpu,end_to_end,32,4,96,14,ON,8299,0.116631,274.368918,0.504127,D_augmentation


## 11. Compute-only vs end-to-end

Cùng một cấu hình, đo hai kiểu để biết CPU/DataLoader/transfer có phải bottleneck hay không.

In [15]:
COMPARE_CFG = dict(
    batch_size=CUR_BATCH,
    num_workers=CUR_WORKERS,
    img_size=CUR_SIZE,
    cpu_threads=CUR_THREADS,
    use_augmentation=True,
)

r_compute = benchmark_compute(**COMPARE_CFG, warmup_batches=10, measure_batches=100)
r_e2e = benchmark_end_to_end(**COMPARE_CFG, warmup_batches=10, measure_batches=60)

compare_df = pd.DataFrame([r_compute, r_e2e])
display(compare_df[
    ["mode", "batch_size", "num_workers", "img_size", "cpu_threads",
     "images_per_sec", "estimated_epoch_min"]
])

compute_ips = r_compute["images_per_sec"]
e2e_ips = r_e2e["images_per_sec"]

print("Compute-only throughput:", round(compute_ips, 2), "images/s")
print("End-to-end throughput:", round(e2e_ips, 2), "images/s")
print("E2E / compute-only:", round(e2e_ips / compute_ips * 100, 1), "%")

,mode,batch_size,num_workers,img_size,cpu_threads,images_per_sec,estimated_epoch_min
0,compute_only,32,4,96,14,531.169194,0.260400
1,end_to_end,32,4,96,14,265.372551,0.521217


Compute-only throughput: 531.17 images/s
End-to-end throughput: 265.37 images/s
E2E / compute-only: 50.0 %


## 12. Benchmark E — final candidate configurations

Các ứng viên được re-test với cùng warm-up/measurement. Kết luận cuối dựa trên **end-to-end throughput**.

In [16]:
candidate_specs = [
    ("E1_b16_w2_s128", 16, 2, 128),
    ("E2_b32_w0_s128", 32, 0, 128),
    ("E3_b32_w2_s128", 32, 2, 128),
    ("E4_b64_w2_s128", 64, 2, 128),
    ("E5_b128_w0_s128", 128, 0, 128),
    ("E6_b16_w2_s96", 16, 2, 96),
    ("E7_b32_w2_s96", 32, 2, 96),
    ("E8_b64_w2_s96", 64, 2, 96),
    ("E9_b16_w2_s160", 16, 2, 160),
    ("E10_b32_w2_s160", 32, 2, 160),
]

candidate_rows = []
for name, batch, workers, size in candidate_specs:
    print(
        f"Running {name} | batch={batch}, workers={workers}, "
        f"threads={CUR_THREADS}, img={size}"
    )
    try:
        r = benchmark_end_to_end(
            batch_size=batch, num_workers=workers, img_size=size,
            cpu_threads=CUR_THREADS, warmup_batches=10,
            measure_batches=100, use_augmentation=True
        )
        r["benchmark"] = "E_final_candidates"
        r["config"] = name
        candidate_rows.append(r)
        print(f"  {r['images_per_sec']:.2f} images/s | {r['estimated_epoch_min']:.2f} min/epoch")
    except Exception as e:
        print("  FAILED:", repr(e))

df_E = pd.DataFrame(candidate_rows)
display(
    df_E[[
        "config", "batch_size", "num_workers", "cpu_threads",
        "img_size", "images_per_sec", "estimated_epoch_min"
    ]].sort_values("images_per_sec", ascending=False)
    if len(df_E) else df_E
)

Running E1_b16_w2_s128 | batch=16, workers=2, threads=14, img=128
  159.02 images/s | 0.87 min/epoch
Running E2_b32_w0_s128 | batch=32, workers=0, threads=14, img=128
  30.05 images/s | 4.60 min/epoch
Running E3_b32_w2_s128 | batch=32, workers=2, threads=14, img=128
  170.81 images/s | 0.81 min/epoch
Running E4_b64_w2_s128 | batch=64, workers=2, threads=14, img=128
  153.63 images/s | 0.90 min/epoch
Running E5_b128_w0_s128 | batch=128, workers=0, threads=14, img=128
  30.03 images/s | 4.61 min/epoch
Running E6_b16_w2_s96 | batch=16, workers=2, threads=14, img=96
  197.46 images/s | 0.70 min/epoch
Running E7_b32_w2_s96 | batch=32, workers=2, threads=14, img=96
  176.92 images/s | 0.78 min/epoch
Running E8_b64_w2_s96 | batch=64, workers=2, threads=14, img=96
  174.39 images/s | 0.79 min/epoch
Running E9_b16_w2_s160 | batch=16, workers=2, threads=14, img=160
  135.87 images/s | 1.02 min/epoch
Running E10_b32_w2_s160 | batch=32, workers=2, threads=14, img=160
  141.93 images/s | 0.97 min/e

,config,batch_size,num_workers,cpu_threads,img_size,images_per_sec,estimated_epoch_min
5,E6_b16_w2_s96,16,2,14,96,197.464924,0.700462
6,E7_b32_w2_s96,32,2,14,96,176.918005,0.781812
7,E8_b64_w2_s96,64,2,14,96,174.385992,0.793164
2,E3_b32_w2_s128,32,2,14,128,170.806566,0.809785
0,E1_b16_w2_s128,16,2,14,128,159.021027,0.869801
3,E4_b64_w2_s128,64,2,14,128,153.630347,0.900321
9,E10_b32_w2_s160,32,2,14,160,141.926527,0.974565
8,E9_b16_w2_s160,16,2,14,160,135.871141,1.017999
1,E2_b32_w0_s128,32,0,14,128,30.045481,4.603576
4,E5_b128_w0_s128,128,0,14,128,30.029156,4.606079


## 13. CPU reference

Không cài lại PyTorch CPU. Chỉ chạy cùng workload trên CPU để có số liệu đối chiếu với XPU.

In [17]:
CPU_REFERENCE_CFG = dict(
    batch_size=32, num_workers=2, img_size=128,
    cpu_threads=8, use_augmentation=True
)

cpu_reference = benchmark_end_to_end(
    **CPU_REFERENCE_CFG, warmup_batches=10, measure_batches=60,
    device=torch.device("cpu")
)

xpu_reference = benchmark_end_to_end(
    **CPU_REFERENCE_CFG, warmup_batches=10, measure_batches=60,
    device=DEVICE
)

reference_df = pd.DataFrame([cpu_reference, xpu_reference])
display(reference_df[
    ["device", "batch_size", "num_workers", "img_size",
     "cpu_threads", "images_per_sec", "estimated_epoch_min"]
])

speedup = xpu_reference["images_per_sec"] / cpu_reference["images_per_sec"]
print("XPU/CPU throughput ratio:", round(speedup, 2), "x")

,device,batch_size,num_workers,img_size,cpu_threads,images_per_sec,estimated_epoch_min
0,cpu,32,2,128,8,72.198753,1.915776
1,xpu,32,2,128,8,164.349679,0.841600


XPU/CPU throughput ratio: 2.28 x


## 14. Final summary + export

Cấu hình cuối được chọn từ Benchmark E theo end-to-end throughput. Thời gian 50 epoch chỉ là ước tính từ training images và throughput benchmark; training thật còn validation, checkpoint, logging, metrics và filesystem.

In [18]:
frames = []
for name, df in [
    ("A_batch_workers", df_A),
    ("B_cpu_threads", df_B),
    ("C_resolution", df_C),
    ("D_augmentation", df_D),
    ("E_final_candidates", df_E),
]:
    if len(df):
        z = df.copy()
        z["benchmark"] = name
        z = z[["benchmark"] + [c for c in z.columns if c != "benchmark"]]
        frames.append(z)

df_all = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if len(df_E):
    final_best = df_E.loc[df_E["images_per_sec"].idxmax()]

    best_name = final_best["config"]
    best_batch = int(final_best["batch_size"])
    best_workers = int(final_best["num_workers"])
    best_threads = int(final_best["cpu_threads"])
    best_size = int(final_best["img_size"])
    best_ips = float(final_best["images_per_sec"])
    best_epoch_min = float(final_best["estimated_epoch_min"])

    epoch50_min = best_epoch_min * 50

    print("\n" + "=" * 72)
    print("FINAL LOCAL TRAINING BENCHMARK")
    print("=" * 72)
    print("PyTorch:", torch.__version__)
    print("Device:", DEVICE)
    print("XPU:", torch.xpu.get_device_name(0))
    print("Training images:", len(preview))
    print("Config:", best_name)
    print("Batch size:", best_batch)
    print("Workers:", best_workers)
    print("CPU threads:", best_threads)
    print("Image size:", best_size)
    print("Augmentation: ON")
    print("End-to-end throughput:", round(best_ips, 2), "images/s")
    print("Estimated epoch:", round(best_epoch_min, 2), "min")
    print("Estimated 50 epochs:", round(epoch50_min, 2), "min")
    print("Estimated 50 epochs:", round(epoch50_min / 60, 2), "hours")
    print("=" * 72)

    out_dir = Path("../outputs/performance_benchmark")
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / "local_cpu_xpu_complete_benchmark_results.csv"
    json_path = out_dir / "local_cpu_xpu_complete_benchmark_results.json"
    txt_path = out_dir / "local_cpu_xpu_final_summary.txt"

    df_all.to_csv(csv_path, index=False)
    df_all.to_json(json_path, orient="records", indent=2)

    summary = f"""
LCC-FASD LOCAL PERFORMANCE BENCHMARK
====================================

PyTorch: {torch.__version__}
Torchvision: {torchvision.__version__}
OS: {platform.platform()}
CPU logical processors: {os.cpu_count()}
RAM GB: {round(psutil.virtual_memory().total / (1024**3), 2)}
XPU available: {torch.xpu.is_available()}
XPU device: {torch.xpu.get_device_name(0)}
Training images: {len(preview)}

FINAL END-TO-END CONFIG
-----------------------
Config: {best_name}
Device: {DEVICE}
Batch size: {best_batch}
Workers: {best_workers}
CPU threads: {best_threads}
Image size: {best_size}
Augmentation: ON
Throughput: {best_ips:.4f} images/s
Estimated epoch: {best_epoch_min:.4f} min
Estimated 50 epochs: {epoch50_min:.4f} min
Estimated 50 epochs: {epoch50_min / 60:.4f} hours

COMPUTE VS END-TO-END
---------------------
Compute-only: {compute_ips:.4f} images/s
End-to-end: {e2e_ips:.4f} images/s
E2E/compute: {e2e_ips / compute_ips * 100:.2f}%

CPU REFERENCE
-------------
CPU throughput: {cpu_reference["images_per_sec"]:.4f} images/s
XPU throughput: {xpu_reference["images_per_sec"]:.4f} images/s
XPU/CPU ratio: {speedup:.4f}x
""".strip()

    txt_path.write_text(summary, encoding="utf-8")

    print("\nSaved:")
    print(csv_path.resolve())
    print(json_path.resolve())
    print(txt_path.resolve())
else:
    print("Không có kết quả Benchmark E; chưa thể chọn cấu hình cuối.")


FINAL LOCAL TRAINING BENCHMARK
PyTorch: 2.14.0+xpu
Device: xpu
XPU: Intel(R) Graphics
Training images: 8299
Config: E6_b16_w2_s96
Batch size: 16
Workers: 2
CPU threads: 14
Image size: 96
Augmentation: ON
End-to-end throughput: 197.46 images/s
Estimated epoch: 0.7 min
Estimated 50 epochs: 35.02 min
Estimated 50 epochs: 0.58 hours

Saved:
/home/dainguyen1506/Documents/face-anti-spoofing/outputs/performance_benchmark/local_cpu_xpu_complete_benchmark_results.csv
/home/dainguyen1506/Documents/face-anti-spoofing/outputs/performance_benchmark/local_cpu_xpu_complete_benchmark_results.json
/home/dainguyen1506/Documents/face-anti-spoofing/outputs/performance_benchmark/local_cpu_xpu_final_summary.txt


## 15. Khi gửi kết quả

Sau khi **Run All**, gửi cho tôi:
1. Output của cell **14 — FINAL LOCAL TRAINING BENCHMARK**.
2. Nếu có `FAILED`, gửi dòng lỗi tương ứng.
3. Nếu muốn gọn, gửi file `local_cpu_xpu_final_summary.txt`.

**Không cần train 50 epoch và không cần tự sửa cấu hình sau benchmark.**